In [80]:
import json
import os
import glob
import numpy as np
from PIL import Image, ImageDraw
from pathlib import Path
from collections import defaultdict
import pandas as pd
import numpy as np
import cv2
from pathlib import Path
from torch.utils.data import Dataset
from transformers import DetrImageProcessor, DetrForObjectDetection
import torch
import requests

c:\Users\denni\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Car Parts Prediction

In [64]:
parts_root = "C:/Users/denni/Documents/MSDS_MSU/ao-damage-estimation/archive/Car_damages_dataset/File1"

img_dir = os.path.join(parts_root, "img")
ann_dir = os.path.join(parts_root, "ann")

image_files = sorted(os.listdir(img_dir))
ann_files = sorted(os.listdir(ann_dir))

In [78]:
# Finding all of the different part classes
unique_classes = set()

for json_file in os.listdir(ann_dir):
    if json_file.endswith(".json"):
        with open(os.path.join(ann_dir, json_file), "r") as f:
            data = json.load(f)
        for obj in data["objects"]:
            unique_classes.add(obj["classTitle"])

num_label = 0
class_mapping = {}
for part in unique_classes:
    class_mapping[num_label] = part
    num_label += 1


In [65]:
def img_to_json(path):

    path_str = str(path)
    return path_str + ".json"

path_dict = []

for img_path in image_files:
    temp = {}
    temp["image_path"] = img_path
    temp["json_path"] = img_to_json(img_path)
    path_dict.append(temp)

parts_df = pd.DataFrame(path_dict)

In [66]:
# example_file = ann_files[0]
# ex_path = ann_dir + "\\" + example_file


# with open(ex_path, "r") as f:
#     data = json.load(f)

# print(type(data))          # What type is it?
# print(data.keys())

# len(data["objects"])

In [67]:
def parse_annotation(json_name):
    """
    Return a structured dictionary of the form 
    {"size": (h, w), "annotations": [{"class": "Hood", "polygon": [[x1,y1], [x2,y2], ...]}, ...]}
    """

    output_dict = {}
    ex_path = ann_dir + "\\" + json_name

    with open(ex_path, "r") as f:
        data = json.load(f)

    size_dict = data["size"]
    h = size_dict["height"]
    w = size_dict["width"]
    
    output_dict["size"] = (h, w)

    objects_ls = data["objects"]

    annotations = []
    for curr in objects_ls:

        curr_part_dict = {"class": curr["classTitle"], "polygon": curr["points"]["exterior"]}

        annotations.append(curr_part_dict)

    output_dict["annotations"] = annotations

    return output_dict


In [71]:
parts_df["parsed_annotations"] = parts_df["json_path"].apply(parse_annotation)


### Dataset Class

Steps:

Load the image from disk

Generate the segmentation mask from the parsed annotation (using cv2.fillPoly to draw the polygons)

Run the image quality validation gate (reject or flag bad images)

Apply angle correction if needed

Normalize the image

Return the (image, mask) pair

In [ ]:
class CarPartsDataset(Dataset):
    def __init__(self):
        """
        Args:
            dataframe: your DataFrame with image filenames and parsed annotations
            img_dir: path to the folder containing the images
            class_names: list of all unique part names, e.g. ["Hood", "Fender", ...]
                         (needed to map each class to a numeric label)
        """